In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install flask-ngrok
!pip install torchvision
!pip install opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 846.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 42.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitl

In [4]:
!cp -r /content/drive/MyDrive/cars_train .
!cp -r /content/drive/MyDrive/cars_test .
!cp -r /content/drive/MyDrive/car_devkit .

In [7]:
!ls /content/drive/MyDrive/car_devkit/devkit


cars_meta.mat	     cars_train_annos.mat  README.txt
cars_test_annos.mat  eval_train.m	   train_perfect_preds.txt


In [1]:
# ============================================
# Step 1: ライブラリ読み込み + パス定義
# ============================================

import os
import scipy.io
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

# パス設定
train_img_dir = '/content/drive/MyDrive/cars_train/cars_train'
test_img_dir = '/content/drive/MyDrive/cars_test/cars_test'
anno_path = '/content/drive/MyDrive/car_devkit/devkit/cars_train_annos.mat'

# ============================================
# Step 2: ラベル読み込みとデータセット定義
# ============================================
data = scipy.io.loadmat(anno_path)['annotations'][0]

class StanfordCarsDataset(Dataset):
    def __init__(self, data, img_dir, transform=None):
        self.data = data
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        img_name = item[5][0]  # fname
        label = int(item[4][0]) - 1  # class (1-indexed -> 0-indexed)
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)
        return image, label

# ============================================
# Step 3: 前処理定義とデータローダ作成
# ============================================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = StanfordCarsDataset(data, train_img_dir, transform)
train_dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# ============================================
# Step 4: モデル準備（EfficientNetB2）
# ============================================
num_classes = len(set([int(item[4][0]) for item in data]))

model = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# ============================================
# Step 5: 学習
# ============================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in train_dataloader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    acc = correct / total
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_dataloader):.4f}, Accuracy: {acc*100:.2f}%")

# ============================================
# Step 6: モデル保存
# ============================================
torch.save(model.state_dict(), '/content/drive/MyDrive/efficientnetb2_car_model.pth')

/tmp/ipython-input-1-2447528223.py:59: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  num_classes = len(set([int(item[4][0]) for item in data]))
/tmp/ipython-input-1-2447528223.py:36: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  label = int(item[4][0]) - 1  # class (1-indexed -> 0-indexed)


Epoch [1/5], Loss: 5.0155, Accuracy: 5.32%
Epoch [2/5], Loss: 3.7133, Accuracy: 29.59%
Epoch [3/5], Loss: 2.3451, Accuracy: 57.10%
Epoch [4/5], Loss: 1.4055, Accuracy: 75.66%
Epoch [5/5], Loss: 0.8404, Accuracy: 85.94%


In [7]:
import torch
from torchvision import models, transforms
from PIL import Image
import scipy.io

# ① クラス名の読み込み
meta_path = '/content/drive/MyDrive/car_devkit/devkit/cars_meta.mat'
meta = scipy.io.loadmat(meta_path)
class_names = [c[0] for c in meta['class_names'][0]]  # 0-indexedで196クラス

# ② モデル構築・読み込み
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.efficientnet_b2(weights=None)
model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, 196)
model.load_state_dict(torch.load('/content/drive/MyDrive/efficientnetb2_car_model.pth'))
model = model.to(device)
model.eval()

# ③ 推論対象画像の前処理
img_path = '/content/drive/MyDrive/2013-ford-expedition-king-ranch.jpg'  # 推論したい画像のパス
image = Image.open(img_path).convert('RGB')
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
input_tensor = transform(image).unsqueeze(0).to(device)

# ④ 推論実行と車種名の出力
with torch.no_grad():
    output = model(input_tensor)
    pred_class_id = torch.argmax(output, dim=1).item()  # 0〜195のID
    pred_class_name = class_names[pred_class_id]        # 対応する車種名

print(f"予測クラスID: {pred_class_id}")
print(f"予測車種名: {pred_class_name}")


予測クラスID: 108
予測車種名: Ford Expedition EL SUV 2009
